<a href="https://colab.research.google.com/github/Arpit-Rathore-603/Food_Delivery_and_Order_Analysis_HCL/blob/main/Copy_of_Food_delivery_and__Order_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import os
import pandas as pd

NUM_COLS = [
    "Customer_Age",
    "Delivery_Time_Mins",
    "Delivery_Fee",
    "Order_Value",
    "Customer_Rating",
]
CAT_COLS = ["Meal_Type", "Order_Channel", "Order_Status"]


# 1. Load CSV file (with Colab upload fallback if not found)
def load_orders(file_path="food_orders_dataset.csv"):
  if not os.path.exists(file_path):
    try:
      from google.colab import files

      uploaded = files.upload()
      file_path = next(iter(uploaded))
    except ImportError:
      pass
  return pd.read_csv(file_path)


# 2. Clean duplicates and fill missing values
def clean_orders(data):
  data = data.copy()
  print("Original orders:", len(data))

  data = data.drop_duplicates()
  print("Unique orders after removing duplicates:", len(data))

  for col in NUM_COLS:
    if col in data.columns:
      data[col] = pd.to_numeric(data[col], errors="coerce")
      med_val = data[col].median()
      data[col] = data[col].fillna(med_val)

  for col in CAT_COLS:
    if col in data.columns:
      data[col] = data[col].fillna("Unknown")

  return data.reset_index(drop=True)


# 3. Add Speed_Group and Risk_Level using simple loops and if/else
def add_order_labels(data):
  data = data.copy()
  speed_groups = []
  risk_levels = []

  for i in range(len(data)):
    mins = data.iloc[i]["Delivery_Time_Mins"]
    fee = data.iloc[i]["Delivery_Fee"]
    rating = data.iloc[i]["Customer_Rating"]

    # Delivery speed tier
    if mins <= 30:
      speed_groups.append("Fast")
    elif mins <= 45:
      speed_groups.append("Standard")
    else:
      speed_groups.append("Slow")

    # Simple risk score from slow delivery, low rating, and high fee
    risk_points = 0
    if mins >= 45:
      risk_points = risk_points + 1
    if rating < 3.5:
      risk_points = risk_points + 1
    if fee >= 60:
      risk_points = risk_points + 1

    if risk_points >= 2:
      risk_levels.append("High Risk")
    elif risk_points == 1:
      risk_levels.append("Moderate Risk")
    else:
      risk_levels.append("Low Risk")

  data["Speed_Group"] = speed_groups
  data["Risk_Level"] = risk_levels
  return data


# 4. Print summary using a dictionary, groupby averages + Total Avg, and a set
def show_order_summary(data):
  status_count = {}
  for status in data["Order_Status"]:
    if status in status_count:
      status_count[status] = status_count[status] + 1
    else:
      status_count[status] = 1

  print("\nOrder Status Counts:")
  for key, value in status_count.items():
    print(f"  {key}: {value}")

  metric_cols = [
      "Delivery_Time_Mins",
      "Delivery_Fee",
      "Order_Value",
      "Customer_Rating",
  ]
  print("\nAverages by Order Status:")
  summary_table = data.groupby("Order_Status")[metric_cols].mean().round(2)
  print(summary_table)

  total_avg = data[metric_cols].mean()
  total_line = (
      "Total Avg (All Orders) -> "
      f"Delivery_Time_Mins: {total_avg['Delivery_Time_Mins']:.2f} | "
      f"Delivery_Fee: {total_avg['Delivery_Fee']:.2f} | "
      f"Order_Value: {total_avg['Order_Value']:.2f} | "
      f"Customer_Rating: {total_avg['Customer_Rating']:.2f}"
  )
  print(total_line)

  high_risk_orders = set(data[data["Risk_Level"] == "High Risk"]["Order_ID"])
  slow_orders = set(data[data["Speed_Group"] == "Slow"]["Order_ID"])
  both_orders = high_risk_orders.intersection(slow_orders)

  print("\nSet Comparison:")
  print("  High Risk orders:", len(high_risk_orders))
  print("  Slow delivery orders:", len(slow_orders))
  print("  Both High Risk & Slow:", len(both_orders))

  data.to_csv("cleaned_food_orders.csv", index=False)
  with open("food_order_summary.txt", "w") as f:
    f.write(f"Total unique orders: {len(data)}\n")
    f.write(total_line + "\n")
    f.write(f"High Risk orders: {len(high_risk_orders)}\n")
    f.write(f"Slow delivery orders: {len(slow_orders)}\n")
    f.write(f"Both High Risk & Slow: {len(both_orders)}\n")
  print("\nSaved 'cleaned_food_orders.csv' and 'food_order_summary.txt'.")


raw_orders = load_orders("food_orders_dataset.csv")
cleaned_orders = clean_orders(raw_orders)
final_orders = add_order_labels(cleaned_orders)
show_order_summary(final_orders)

Saving food_orders_dataset.csv to food_orders_dataset.csv
Original orders: 121
Unique orders after removing duplicates: 120

Order Status Counts:
  Issue / Refund: 59
  Delayed: 41
  Delivered On-Time: 20

Averages by Order Status:
                   Delivery_Time_Mins  Delivery_Fee  Order_Value  \
Order_Status                                                       
Delayed                         64.44         50.22       724.91   
Delivered On-Time               27.55         51.54       793.52   
Issue / Refund                  40.80         49.73       754.82   

                   Customer_Rating  
Order_Status                        
Delayed                       3.69  
Delivered On-Time             4.32  
Issue / Refund                3.41  
Total Avg (All Orders) -> Delivery_Time_Mins: 46.67 | Delivery_Fee: 50.20 | Order_Value: 751.05 | Customer_Rating: 3.66

Set Comparison:
  High Risk orders: 50
  Slow delivery orders: 67
  Both High Risk & Slow: 39

Saved 'cleaned_food_orders